# Zlab Cloud · Earlier braking with a stronger ball-contact penalty
Select Kaggle T4, enable Internet, and run all cells. Train from the original 1199 with earlier braking, lateral stability during the stop, and a 40-unit terminal cost for touching the ball. Checkpoint 1499 is a reference. All policies are evaluated with the same 2.8 m/s cap, earlier braking, and five-second hold.


In [ ]:
import os
from pathlib import Path
import subprocess
import tempfile

repo = Path(tempfile.mkdtemp(prefix="zlab-cloud-")) / "Zlab-cloud"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/hafizadha/Zlab-cloud.git", str(repo),
], check=True)
project = repo / "Zlab_Gym"
launcher = project / "train_ball_stability.sh"
if not launcher.is_file():
    raise RuntimeError("This checkout does not contain the ball stability launcher.")
subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], check=True)


import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[huggingface]", "onnx"], cwd=project, check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator before training.")

In [ ]:
# Set this to True to back up checkpoints to Hugging Face.
# For uploads, add a write token in Kaggle Secrets.
UPLOAD_TO_HF = False

if UPLOAD_TO_HF:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["ZLAB_HF_REPO_ID"] = "hafiz24/zlab-checkpoints"


In [ ]:
PILOT_BLOCKS = 1
UPDATES_PER_BLOCK = 300
ENVS_PER_GPU = 512
FOOT_FRICTION_RANGE = (0.35, 1.15)  # Broad simulation range until you measure your turf.
START_CHECKPOINT = ""  # Optional absolute path to a saved stability checkpoint.
HF_START_FILE = ""  # e.g. checkpoints/<run>/model_1400.pt
WORK_DIR = Path("/kaggle/working/zlab-robocup-contact-avoidance-pilot")
progress = WORK_DIR / "progress.json"

if START_CHECKPOINT and HF_START_FILE:
    raise ValueError("Choose one checkpoint source.")
if START_CHECKPOINT:
    checkpoint = Path(START_CHECKPOINT)
elif HF_START_FILE:
    from huggingface_hub import hf_hub_download

    checkpoint = Path(hf_hub_download(
        repo_id="hafiz24/zlab-checkpoints", filename=HF_START_FILE, repo_type="model"
    ))
else:
    checkpoint = project / "checkpoints/model_1199.pt"

subprocess.run([
    sys.executable, "scripts/train_stability_blocks.py", str(checkpoint),
    "--work-dir", str(WORK_DIR), "--pilot-blocks", str(PILOT_BLOCKS),
    "--lateral-pilot", "--early-braking",
    "--finishing-control", str(project / "checkpoints/model_1499.pt"),
    "--pilot-seeds", "97103", "108173", "--pilot-train-seed", "119027",
    "--updates", str(UPDATES_PER_BLOCK), "--num-envs", str(ENVS_PER_GPU),
], cwd=project, env=dict(
    os.environ, ZLAB_USE_ALL_GPUS="1",
    ZLAB_FOOT_FRICTION_MIN=str(FOOT_FRICTION_RANGE[0]),
    ZLAB_FOOT_FRICTION_MAX=str(FOOT_FRICTION_RANGE[1]),
), check=True)

In [ ]:
# Selected checkpoint and paired evaluation reports.
import json

if progress.is_file():
    state = json.loads(progress.read_text())
    print("Selected checkpoint:", state["checkpoint"])
    if state["stop_reason"] == "no_improvement":
        print("Training stopped after a block with no improvement on both seeds.")
    comparison = state.get("latest_comparison")
    if comparison:
        print("Comparison report:", comparison)
        print(Path(comparison).read_text())